# Revisão manual de erros (Task 2)

Mostra, painel a painel, **imagem | gabarito | predição** para os 25 piores painéis e 25 aleatórios de um `panels.csv` gerado por `analyze_errors.py`.

A marcação é feita no `review.csv` (abra numa planilha ou edite a célula do fim). Colunas a preencher: `categorias` (separadas por `;`), `gt_suspeito` (sim/não) e `observacao`.

Categorias (Parte 2 do `docs/plano_analise_erros.md`): `painel_faltando`, `tabela_invalida`, `colunas_divergentes`, `amostragem_linhas`, `cabecalho_desalinhado`, `escala_magnitude`, `grade_regular_inventada`, `numero_nao_numerico`. Erro que não cabe em nenhuma vai em `observacao`.

A coluna `categorias_regra` mostra o que as regras automáticas acharam; marque o que **você** vê, sem copiar. A última célula compara as duas coisas.

In [ ]:
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PANELS_CSV = ROOT / "outputs/analise_erros/lora3b/panels.csv"
REVIEW_CSV = PANELS_CSV.with_name("review.csv")
METADATA = ROOT / "data/raw/validation_metadata.jsonl"
IMAGES_DIR = ROOT / "data/raw/images"
SEED = 42
N_WORST, N_RANDOM = 25, 25

In [ ]:
import html
import sys

import pandas as pd
from IPython.display import HTML, display
from PIL import Image

sys.path.insert(0, str(ROOT))
from src.analysis.reference import load_panels

df = pd.read_csv(PANELS_CSV, keep_default_na=False)
refs = {(p.sample_id, p.panel_id): p for p in load_panels(str(METADATA))}

worst = df.nsmallest(N_WORST, "weighted")
rest = df.drop(worst.index)
sample = pd.concat([
    worst.assign(origem="pior"),
    rest.sample(min(N_RANDOM, len(rest)), random_state=SEED).assign(origem="aleatorio"),
]).reset_index(drop=True)
print(f"{len(sample)} painéis para revisar (de {len(df)})")

In [ ]:
def show(i):
    row = sample.iloc[i]
    ref = refs[(row.sample_id, row.panel_id)]
    img = Image.open(IMAGES_DIR / ref.image_file)
    if ref.bbox:
        b = ref.bbox
        img = img.crop((b["x"], b["y"], b["x"] + b["width"], b["y"] + b["height"]))
    display(HTML(
        f"<h4>[{i}] {html.escape(row.sample_id)} / painel {row.panel_id} ({row.origem})</h4>"
        f"<p>{html.escape(row.figure_type)} · weighted {row.weighted:.1f} (RMS {row.rms:.1f}, TEDS {row.teds:.1f}) · "
        f"gabarito {row.gt_rows}×{row.gt_cols}, predição {row.pred_rows}×{row.pred_cols}</p>"
    ))
    display(img)
    display(HTML(
        "<table><tr><th>Gabarito</th><th>Predição</th></tr><tr>"
        f"<td style='vertical-align:top'><pre>{html.escape(row.gt_table)}</pre></td>"
        f"<td style='vertical-align:top'><pre>{html.escape(row.pred_table) or '(sem tabela)'}</pre></td>"
        "</tr></table>"
    ))

for i in range(len(sample)):
    show(i)

In [ ]:
# Cria o review.csv na primeira vez; depois só relê o que já foi marcado.
if REVIEW_CSV.exists():
    review = pd.read_csv(REVIEW_CSV, keep_default_na=False)
else:
    review = sample[["sample_id", "panel_id", "origem", "weighted"]].assign(
        categorias_regra=sample["categorias"], categorias="", gt_suspeito="", observacao=""
    )
    review.to_csv(REVIEW_CSV, index=False)
print(REVIEW_CSV)
review

In [ ]:
# Depois de revisar: quanto cada regra acerta frente à sua marcação (precision = das vezes que a regra disparou, quantas estavam certas).
from src.analysis.calibration import rule_precision

rule_precision(str(PANELS_CSV), str(REVIEW_CSV))